# Instagram Influencer and Brand Dataset: first audit

Run the cells in order. Nothing here extracts the 3 GB JSON archive or touches the image archives.
Change BASE in the first cell if your Drive shortcut has a different name.

In [4]:
from google.colab import drive
drive.mount('/content/drive')

import os, re, json, shutil, random, zipfile
import numpy as np
import pandas as pd

BASE = '/content/drive/MyDrive/Influencer brand dataset'

if not os.path.exists(f'{BASE}/post_info.txt'):
    # Fallback: search again, this time following shortcuts
    BASE = None
    for dirpath, dirnames, filenames in os.walk('/content/drive/MyDrive', followlinks=True):
        if 'post_info.txt' in filenames:
            BASE = dirpath
            break
        if dirpath.count('/') - 3 >= 2:   # do not descend more than two levels
            dirnames[:] = []

print('BASE =', BASE)
assert BASE is not None, 'post_info.txt still not reachable; see the note below'
print(sorted(os.listdir(BASE))[:8], '...')

LOCAL = '/content/data'
os.makedirs(LOCAL, exist_ok=True)

# Copy the light-tier files to local disk once (reading through the Drive mount is slow)
for f in ['post_info.txt', 'json_files.zip', 'profiles_influencers.zip', 'profiles_brands.zip']:
    dst = f'{LOCAL}/{f}'
    if not os.path.exists(dst):
        print('copying', f)
        shutil.copy(f'{BASE}/{f}', dst)
print(os.listdir(LOCAL))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
BASE = /content/drive/MyDrive/Influencer brand dataset
['README.docx', 'img_fi01.zip', 'img_fi02.zip', 'img_fi03.zip', 'img_fi04.zip', 'img_fi05.zip', 'img_fi06.zip', 'img_fi07.zip'] ...
['json_files.zip', 'post_info.txt', 'profiles_brands.zip', 'profiles_influencers.zip']


## 1. post_info.txt: structure, label balance, coverage

In [5]:
cols = ['post_id', 'username', 'sponsored', 'json_file', 'image_files']
posts = pd.read_csv(f'{LOCAL}/post_info.txt', sep='\t', header=None, names=cols,
                    dtype=str, quoting=3, keep_default_na=False, on_bad_lines='warn')
posts['post_id'] = pd.to_numeric(posts['post_id'], errors='coerce')
posts['sponsored'] = pd.to_numeric(posts['sponsored'], errors='coerce')
posts['n_images'] = np.where(posts['image_files'].str.strip() == '[]', 0,
                             posts['image_files'].str.count(',') + 1)

print('rows:', len(posts))
print('post_id contiguous 0..N-1:', bool((posts['post_id'].to_numpy() == np.arange(len(posts))).all()))
print('\nsponsored label counts:\n', posts['sponsored'].value_counts(dropna=False))
print('sponsored rate: %.4f' % posts['sponsored'].mean())
print('\nunique usernames:', posts['username'].nunique())
print('posts per user:\n', posts.groupby('username').size().describe())
print('\nduplicate json_file names:', posts['json_file'].duplicated().sum())
print('\nimages per post:\n', posts['n_images'].value_counts().sort_index().head(12))

rows: 1601074
post_id contiguous 0..N-1: True

sponsored label counts:
 sponsored
0    1379364
1     221710
Name: count, dtype: int64
sponsored rate: 0.1385

unique usernames: 38113
posts per user:
 count    38113.000000
mean        42.008606
std         49.601017
min          1.000000
25%          9.000000
50%         24.000000
75%         56.000000
max        300.000000
dtype: float64

duplicate json_file names: 0

images per post:
 n_images
0          70
1     1368213
2      125095
3       51609
4       23270
5       12005
6        6860
7        4192
8        2941
9        2438
10       4377
16          1
Name: count, dtype: int64


In [6]:
zj = zipfile.ZipFile(f'{LOCAL}/json_files.zip')
zip_names = {n.split('/')[-1] for n in zj.namelist() if n.endswith('.json')}
in_zip = posts['json_file'].isin(zip_names)
print('posts whose JSON exists in the zip: %.5f' % in_zip.mean())
print('JSON files in zip not listed in post_info:', len(zip_names - set(posts['json_file'])))

posts whose JSON exists in the zip: 1.00000
JSON files in zip not listed in post_info: 0


## 2. Label check: which JSON fields track the sponsorship label?

This decides whether a caption model would only be re-discovering a hashtag or field that produced the label.

In [7]:
def read_json(name):
    with zj.open('json/' + name) as f:
        return json.load(f)

n_each = 300
pos = posts[posts['sponsored'] == 1]
neg = posts[posts['sponsored'] == 0]
samp = pd.concat([pos.sample(min(n_each, len(pos)), random_state=42),
                  neg.sample(min(n_each, len(neg)), random_state=42)])

recs = []
for r in samp.itertuples():
    try:
        recs.append((int(r.sponsored), read_json(r.json_file)))
    except Exception:
        recs.append((int(r.sponsored), None))
ok = [(y, j) for y, j in recs if j is not None]
print('sampled:', len(recs), '| unreadable:', len(recs) - len(ok))

def nonempty(v):
    if v is None:
        return False
    if isinstance(v, bool):
        return v
    if isinstance(v, (int, float)):
        return v != 0
    if isinstance(v, (str, list, dict)):
        return len(v) > 0
    return True

def flatten(d, prefix='', depth=0, out=None):
    out = {} if out is None else out
    for k, v in d.items():
        p = f'{prefix}{k}'
        if isinstance(v, dict) and depth < 2 and v:
            flatten(v, p + '.', depth + 1, out)
        else:
            out[p] = v
    return out

flat = [(y, flatten(j)) for y, j in ok]
allkeys = sorted({k for _, f in flat for k in f})
n1 = sum(1 for y, _ in flat if y == 1)
n0 = sum(1 for y, _ in flat if y == 0)

rows = []
for k in allkeys:
    s = sum(nonempty(f.get(k)) for y, f in flat if y == 1) / max(n1, 1)
    o = sum(nonempty(f.get(k)) for y, f in flat if y == 0) / max(n0, 1)
    rows.append((k, round(s, 3), round(o, 3), round(abs(s - o), 3)))
scan = pd.DataFrame(rows, columns=['field', 'sponsored_rate', 'organic_rate', 'gap'])
print('\nTotal distinct field paths:', len(scan))
scan.sort_values('gap', ascending=False).head(25)

sampled: 600 | unreadable: 0

Total distinct field paths: 51


,field,sponsored_rate,organic_rate,gap
15,edge_media_to_tagged_user.edges,0.183,0.123,0.060
11,edge_media_to_comment.edges,0.197,0.137,0.060
5,display_resources,0.200,0.147,0.053
46,viewer_can_reshare,0.200,0.147,0.053
39,owner.username,0.200,0.147,0.053
37,owner.profile_pic_url,0.200,0.147,0.053
43,thumbnail_resources,0.800,0.853,0.053
44,thumbnail_src,0.800,0.853,0.053
45,tracking_token,0.200,0.147,0.053
31,owner.full_name,0.200,0.147,0.053


In [8]:
def caption(j):
    try:
        return j['edge_media_to_caption']['edges'][0]['node']['text']
    except Exception:
        return ''

pat = re.compile(r'#(ad|sponsored|sponsor|paidpartnership|gifted|collab|partner|brandambassador|sp)\b', re.I)

chk = pd.DataFrame({'sponsored': [y for y, j in ok],
                    'disclosure_hashtag': [bool(pat.search(caption(j))) for y, j in ok]})
print('Share of posts containing a disclosure-style hashtag, by label:')
print(pd.crosstab(chk['sponsored'], chk['disclosure_hashtag'], normalize='index').round(3))

print('\n--- 5 sponsored captions ---')
for y, j in [x for x in ok if x[0] == 1][:5]:
    print(repr(caption(j)[:300]), '\n')
print('--- 5 organic captions ---')
for y, j in [x for x in ok if x[0] == 0][:5]:
    print(repr(caption(j)[:300]), '\n')

Share of posts containing a disclosure-style hashtag, by label:
disclosure_hashtag  False  True 
sponsored                       
0                   0.970  0.030
1                   0.027  0.973

--- 5 sponsored captions ---
'BE LIMITLESS 🌚🚀💫 You know how people say sky’s the limit? Well, why does it have to be? If you put your mind, soul, sweat & tears into what you love, THERE’S NO LIMIT TO WHAT YOU CAN ACCOMPLISH 💥 But it all starts with your mentality, so repeat after me: I am capable. I can achieve whatever I put my' 

'My work wear and glasses look book just went live (link in stories). A lot of you have been asking about my glasses so I decided to add a mini haul and a few tips into the video as well! @glassesusa #ad #glassesusa 🤓' 

'ITS SOUP SEASON 👏🏻👏🏻 if you’ve been following me for a while you know how much im obsessed with @panerabread soups. everyyyytime i go, i pull apart my baguette (if you don’t pick the baguette as your side item what are you even doing?) + let it so

## 3. Influencer profiles

In [9]:
zp = zipfile.ZipFile(f'{LOCAL}/profiles_influencers.zip')
print('sample member names:', [n for n in zp.namelist()[:6]])

def parse_profiles(z):
    rows = []
    for n in z.namelist():
        if n.endswith('/'):
            continue
        txt = z.read(n).decode('utf-8', 'replace').strip('\n')
        parts = txt.split('\t')
        uname = os.path.basename(n)
        if len(parts) < 11:
            rows.append({'username': uname, 'malformed': True})
            continue
        rows.append({'username': uname, 'name': parts[0], 'followers': parts[1],
                     'followees': parts[2], 'n_posts': parts[3], 'url': parts[4],
                     'flag': parts[5], 'category': parts[6],
                     'bio': '\t'.join(parts[7:-3]), 'email': parts[-3],
                     'phone': parts[-2], 'pic': parts[-1], 'malformed': False})
    return pd.DataFrame(rows)

inf = parse_profiles(zp)
for c in ['followers', 'followees', 'n_posts']:
    inf[c] = pd.to_numeric(inf[c], errors='coerce')

print('profiles:', len(inf), '| malformed:', int(inf['malformed'].sum()))
print('\ntop categories:\n', inf['category'].value_counts(dropna=False).head(20))
print('\n', inf[['followers', 'followees', 'n_posts']].describe().round(1))
print('\nusername examples:', inf['username'].head(5).tolist())

post_users = set(posts['username'])
prof_users = set(inf['username'])
print('\nusers in post_info:', len(post_users), '| with a profile file:', len(post_users & prof_users))

sample member names: ['users_influencers_SPOD/', 'users_influencers_SPOD/fodmap_friendly_eating', 'users_influencers_SPOD/paileaa', 'users_influencers_SPOD/a12team', 'users_influencers_SPOD/ilbulgaro89', 'users_influencers_SPOD/joililikoi']
profiles: 38113 | malformed: 0

top categories:
 category
Creators & Celebrities                         24487
NULL                                            6017
Publishers                                      1830
Personal Goods & General Merchandise Stores     1510
General Interest                                1421
Non-Profits & Religious Organizations            562
Transportation & Accomodation Services           342
Home Services                                    306
Business & Utility Services                      288
Home Goods Stores                                252
Lifestyle Services                               233
Local Events                                     179
Food & Personal Goods                            171
Professional